# Logistic Regression with DeepNetts (JSR 381)

DeepNetts is the reference implementation of **JSR 381**, the Visual Recognition
and Machine Learning specification. Yes, Java has a *standardized* ML API, and
this is the one library here you code against a spec rather than a single
vendor's classes.

It's also a neural-network library, which makes it the most conceptually honest
place to see what logistic regression actually *is*: a network with an input
layer and a **single sigmoid output neuron, no hidden layer**. That is logistic
regression, exactly. When people say "logistic regression is the simplest neural
network," this i what they mean.

The shared `.java` files do the data work. This notebook does the DeepNetts part:
**adapt our table into a `TabularDataSet`, build the one-neuron network, train,
and read the result.**

In [ ]:
%%loadFromPOM
<dependency>
    <groupId>com.fasterxml.jackson.dataformat</groupId>
    <artifactId>jackson-dataformat-csv</artifactId>
    <version>2.17.2</version>
</dependency>
<dependency>
    <groupId>com.deepnetts</groupId>
    <artifactId>deepnetts-core</artifactId>
    <version>1.13.2</version>
</dependency>
<dependency>
    <groupId>javax.visrec</groupId>
    <artifactId>visrec-api</artifactId>
    <version>1.0.5</version>
</dependency>

## The shared pipeline

Identical to every other notebook in this post.

In [ ]:
%load shared/Match.java
%load shared/DataLoader.java
%load shared/FormerName.java
%load shared/TeamNames.java
%load shared/EloRating.java
%load shared/RecentForm.java
%load shared/FeatureRow.java
%load shared/FeatureEngineering.java
%load shared/TrainTestSplit.java
%load shared/Metrics.java
%load shared/Predictions.java

## DeepNetts imports

In [ ]:
import deepnetts.data.TabularDataSet;
import deepnetts.net.FeedForwardNetwork;
import deepnetts.net.layers.activation.ActivationType;
import deepnetts.net.loss.LossType;
import deepnetts.util.Tensor;
System.out.println("deepnetts imports ready");

## The DeepNetts adapter (with standardization)

DeepNetts trains on `TabularDataSet`, whose rows are `Item(float[] input,
float[] output)`. Two things differ from the other three libraries.

First, DeepNetts wants `float`, not `double`, so we narrow.

Second (and this is the real lesson) DeepNetts **diverges to `NaN` on our raw
features**. Its backpropagation can't handle `eloDiff` measured in the
hundreds sitting next to rates in `[0, 1]`; the gradients explode on the first
epoch. **Smile, Tribuo, and Weka** all shrugged this off because their solvers are
robust to feature scale. DeepNetts' raw gradient descent is not. So here we
**standardize** the features (z-score: subtract the mean, divide by the standard
deviation), fitting the statistics on the *training set only* and applying them
to test and upcoming, never trainig with data we're meant to predict.

This is the same scaling problem that scrambled the coefficients in the Smile
notebook. but here it's fatal. Standardizing is what it takes to make a neural network behave.

We also call `setColumnNames(...)` so DeepNetts' built-in accuracy check can size
its confusion matrix, without it, training throws (a sharp edge we hit the hard
way). Everything is in one cell so no import lands between creating and using the
`split`/`fe` variables.

In [ ]:
var all = DataLoader.loadAll("/home/jovyan/data/results.csv");
var names = TeamNames.load("/home/jovyan/data/former_names.csv");
var fe = FeatureEngineering.build(all, names);
var split = TrainTestSplit.chronological(fe.played(), 0.8);

String[] featureNames = FeatureRow.featureNames();
int numInputs = featureNames.length;

// --- Standardizer: fit mean/std on TRAIN ONLY, then reuse for test/upcoming ---
double[] mean = new double[numInputs];
double[] std  = new double[numInputs];
for (FeatureRow r : split.train()) {
    double[] f = r.features();
    for (int j = 0; j < numInputs; j++) mean[j] += f[j];
}
for (int j = 0; j < numInputs; j++) mean[j] /= split.train().size();
for (FeatureRow r : split.train()) {
    double[] f = r.features();
    for (int j = 0; j < numInputs; j++) std[j] += (f[j]-mean[j])*(f[j]-mean[j]);
}
for (int j = 0; j < numInputs; j++) {
    std[j] = Math.sqrt(std[j] / split.train().size());
    if (std[j] == 0.0) std[j] = 1.0; // guard against constant columns
}

// scale one feature vector into a float[] DeepNetts can train on
float[] scale(double[] f) {
    float[] out = new float[numInputs];
    for (int j = 0; j < numInputs; j++) out[j] = (float) ((f[j]-mean[j]) / std[j]);
    return out;
}

// Column names = features + target; needed for DeepNetts' evaluator.
String[] columnNames = new String[numInputs + 1];
System.arraycopy(featureNames, 0, columnNames, 0, numInputs);
columnNames[numInputs] = "homeWin";

var trainSet = new TabularDataSet(numInputs, 1);
trainSet.setColumnNames(columnNames);
for (FeatureRow r : split.train()) {
    float label = (r.homeWin() != null && r.homeWin()) ? 1f : 0f;
    trainSet.add(new TabularDataSet.Item(scale(r.features()), new float[]{label}));
}

var testRows = split.test();
var upcomingRows = fe.upcoming();
int[] testY = TrainTestSplit.toY(testRows);
System.out.println("train rows: " + trainSet.size() + "   test rows: " + testRows.size());
System.out.println("feature means (train): " + java.util.Arrays.toString(mean));

## Build the network and train

Input layer of six features → a single sigmoid output, cross-entropy loss. No
hidden layer: this *is* logistic regression, expressed as a neural network.

DeepNetts prints a per-epoch training log to `System.out`. It's informative
interactively but noisy in our notebook because of the learning purpose.

In [ ]:
// DeepNetts prints a per-epoch training log straight to the console (it grabs
// System.out/err when the trainer is built, not via a logger we can mute). So we
// redirect both streams around the whole build-and-train block, then restore.
// Want to watch it converge? Comment out the two setOut/setErr lines.
FeedForwardNetwork net;
double trainingAccuracy;

var realOut = System.out;
var realErr = System.err;
var devNull = new java.io.PrintStream(java.io.OutputStream.nullOutputStream());
System.setOut(devNull);
System.setErr(devNull);
try {
    net = FeedForwardNetwork.builder()
        .addInputLayer(numInputs)
        .addOutputLayer(1, ActivationType.SIGMOID)   // one sigmoid neuron == logistic regression
        .lossFunction(LossType.CROSS_ENTROPY)
        .build();

    var trainer = net.getTrainer();
    trainer.setMaxError(0.01f);
    trainer.setMaxEpochs(30);
    trainer.setLearningRate(0.1f);
    trainer.train(trainSet);
    trainingAccuracy = trainer.getTrainingAccuracy();
} finally {
    System.setOut(realOut);
    System.setErr(realErr);
}
System.out.println("trained — training accuracy " + trainingAccuracy);

## Evaluate

We run each test row through the network, read the single sigmoid output as
P(home win), and score with the shared `Metrics`.

In [ ]:
double[] testProbs = new double[testRows.size()];
for (int i = 0; i < testRows.size(); i++) {
    net.setInput(Tensor.create(1, numInputs, scale(testRows.get(i).features())));
    testProbs[i] = net.getOutput()[0];
}

var metrics = Metrics.from(testProbs, testY);
System.out.println("DeepNetts (JSR 381) logistic regression");
System.out.println(metrics);

## Predict the 2026 World Cup group stage

The 44 matches with no result from the 2026 World Cup group stage. One forward
pass each.

In [ ]:
double[] upcomingProbs = new double[upcomingRows.size()];
for (int i = 0; i < upcomingRows.size(); i++) {
    net.setInput(Tensor.create(1, numInputs, scale(upcomingRows.get(i).features())));
    upcomingProbs[i] = net.getOutput()[0];
}
Predictions.print(upcomingRows, upcomingProbs);

## A note on the standard

We trained DeepNetts through its native API here, which keeps the neural-network
framing visible. JSR 381 *also* defines a higher-level, vendor-neutral interface
(`javax.visrec.ml.classification.NeuralNetClassifier` and friends) that you can
build a model against without naming DeepNetts at all, the standard's whole
point. That portability is the reason to reach for JSR 381 over a single-vendor
library, even though, as we saw, the reference implementation still has its sharp
edges.